# 18 - Travel-Time Weighted Network

Every notebook in this project up to here has measured distance in **hops**: the shortest path
between two stations is the one that passes through the fewest intermediate stops. That is a real
and largely unstated limitation. Two consecutive urban bus stops 200 m apart count exactly the same
as a 40-minute intercity rail leg, so a "short" path in the hop graph can be a long journey, and a
station that looks like a bottleneck under hop counting may be irrelevant once minutes are what
matter.

This notebook removes that limitation. It streams the raw `stop_times.txt` feed once, and for every
consecutive stop pair inside a trip computes the scheduled travel time in seconds. Collecting the
whole distribution per segment gives a **median / p25 / p75 travel time per edge**, which becomes the
edge weight of a new graph, `graph_traveltime.pkl`. That graph is the foundation for notebooks 20-22
(dynamic resilience, demand-weighted criticality, rerouting cost), none of which can produce a
meaningful "detour in minutes" number on a hop graph.

The notebook then asks the question that decides how much of the earlier work has to be re-read:
**does re-weighting change who the critical stations are?** We compare travel-time shortest paths to
hop-count shortest paths on a sample of origin-destination pairs, recompute betweenness on the
travel-time graph, and correlate it with the hop-count betweenness from notebook 04. If the ranking
moves materially, that is a caveat on every centrality-based claim made so far, and we say so.

**Research question:** how much of the network's measured structure is an artefact of using hop count
as distance, and which stations gain or lose importance once edges are weighted by scheduled minutes?

## Inputs

| Path | Produced by | Used for |
|---|---|---|
| `israel-public-transportation/stop_times.txt` | raw GTFS (816 MB, **not in git** - downloaded below) | the travel-time observations |
| `outputs/nb/02_graph_construction/tables/nodes.csv` | notebook **02** | station names, coordinates, region, metro |
| `outputs/nb/02_graph_construction/tables/edges.csv` | notebook **02** | `trip_frequency` per directed segment |
| `outputs/nb/04_centrality_analysis/tables/stop_metrics.csv` | notebook **04** | `approx_betweenness` - the hop-count baseline |

## Must run first

`01_data_preparation` -> `02_graph_construction` -> `04_centrality_analysis`.
Every loader below raises an explicit `FileNotFoundError` naming the notebook to run if an
artifact is missing.

## Outputs (everything under `outputs/nb/18_travel_time_network/`)

| Path | Contents |
|---|---|
| `tables/edges_traveltime.csv` | `from_stop, to_stop, median_travel_seconds, p25_travel_seconds, p75_travel_seconds, trip_frequency, n_observations` - **the contract table other notebooks read** |
| `graph_traveltime.pkl` | undirected `networkx.Graph`; edge attribute `travel_seconds` (also mirrored to `weight`) |
| `traveltime_summary.json` | headline numbers, every discard counter, and the betweenness comparison |
| `tables/discard_reasons.csv` | how many segment observations were dropped and why |
| `tables/path_comparison.csv` | per-pair hop-shortest vs time-shortest path comparison |
| `tables/betweenness_traveltime.csv` | travel-time betweenness next to both hop-count baselines |
| `tables/betweenness_rank_movers.csv` | the stations whose betweenness rank moves most |
| `figures/*.png` | five diagnostic figures |

Nothing outside `outputs/nb/18_travel_time_network/` is written. The report-cited folders
`outputs/tables`, `outputs/figures` and `outputs/rail` are never touched.

## 1. Environment bootstrap

The cell below makes the notebook runnable both on a local checkout and on Google Colab. It defines
`_ensure(...)`, which pip-installs only the packages that are genuinely missing (so re-running the
notebook is cheap), and `find_repo_root()`, which walks up from the current directory looking for the
GTFS folder and, failing that, clones the repository into `/content`. It then sets `REPO`, `DATA` and
`OUT` and creates the notebook output root. Every later cell relies on these three paths, so this
must run first.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## 2. Libraries, stage folders and cost knobs

Standard scientific stack plus the standard-library `csv` module, which is what actually reads the
816 MB feed one row at a time. The stage folder follows the project convention: this notebook owns
`outputs/nb/18_travel_time_network/` and writes nowhere else.

All runtime and all modelling thresholds are gathered here so a grader can change them in one place.
The expensive ones, with honest cost estimates on a 30.5k-node / 51.8k-edge graph:

| Constant | Default | Cost |
|---|---|---|
| the streaming pass over `stop_times.txt` | always runs | **3-6 minutes**, ~150 MB peak RAM for the per-segment duration arrays |
| `RUN_BETWEENNESS` | `True` | set to `False` to skip section 9-11 entirely |
| `K_BETWEENNESS` | `200` sampled sources | **weighted** betweenness needs a Dijkstra per source instead of a BFS: budget **5-15 minutes** for the travel-time run plus **1-3 minutes** for the hop-count control run |
| `N_PATH_PAIRS` | `300` origin-destination pairs | ~1-3 minutes (one bidirectional BFS + one bidirectional Dijkstra per pair) |

The modelling thresholds are `MIN_TRAVEL_SECONDS` and `MAX_TRAVEL_SECONDS`: an observed segment
duration outside `[1 s, 3 h]` is discarded as a data error rather than trusted. Section 6 reports
exactly how many observations each rule removed, so the choice is auditable and not hidden.

In [ ]:
_ensure("pandas", "numpy", "networkx", "matplotlib", "seaborn")

import csv, json, pickle, random, time
from array import array
from collections import defaultdict

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.05)
csv.field_size_limit(10_000_000)   # a few rows in the feed are unusually long

# ---------------- stage folders ----------------
STAGE = OUT / "18_travel_time_network"
TABLES = STAGE / "tables"
FIGURES = STAGE / "figures"
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

# ---------------- modelling thresholds ----------------
MIN_TRAVEL_SECONDS = 1        # a segment must take strictly positive time
MAX_TRAVEL_SECONDS = 3 * 3600 # 3 hours: above this it is a feed error, not a bus ride
MIN_OBSERVATIONS = 1          # segments with fewer valid observations are dropped

# ---------------- cost knobs (see the table above) ----------------
PROGRESS_EVERY = 2_000_000    # progress print interval during the streaming pass
RUN_BETWEENNESS = True        # False -> skip the (slow) betweenness sections 9-11
K_BETWEENNESS = 200           # sampled sources for approximate betweenness
BETWEENNESS_SEED = 42         # same seed for both runs => same sampled source set
N_PATH_PAIRS = 300            # origin-destination pairs in the path comparison
PATH_SEED = 11
RANK_POOL = 500               # rank-mover analysis is restricted to this top-N pool
TOP_N = 15
FIG_DPI = 150

print("pandas", pd.__version__, "| networkx", nx.__version__)
print("stage folder:", STAGE)

## 3. Hebrew label rendering

Stop names in the Israeli GTFS feed are Hebrew, and several figures below print them. Matplotlib does
not implement the Unicode bidirectional algorithm, so right-to-left text comes out reversed and
unreadable. The cell below monkey-patches `matplotlib.text.Text.set_text` once so that any string
containing Hebrew characters is converted to display order via `python-bidi` before it is drawn, and
selects a font that actually has Hebrew glyphs (Arial on Windows, DejaVu Sans everywhere else). It is
idempotent - re-running it will not stack patches. All other text in the notebook is English.

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## 4. Locating earlier stages

Stage folders are resolved by their **two-digit prefix** (`OUT.glob("02*")`), not by an exact slug, so
a renamed folder does not break the chain. `stage_file` raises a `FileNotFoundError` that names the
notebook to run first, because a silent fallback here would produce a graph with no station names and
no baseline to compare against - a failure that would only surface several sections later.

In [ ]:
def stage_dir(prefix, notebook_hint):
    """Resolve a stage folder by its two-digit prefix, e.g. '02' -> 02_graph_construction."""
    matches = sorted(p for p in OUT.glob(f"{prefix}*") if p.is_dir())
    if not matches:
        raise FileNotFoundError(
            f"No stage folder matching '{prefix}*' under {OUT} - "
            f"run notebook {notebook_hint} first."
        )
    return matches[0]


def stage_file(prefix, filename, notebook_hint):
    """Return the path of `filename` inside stage `prefix`, searching sub-folders."""
    root = stage_dir(prefix, notebook_hint)
    direct = root / filename
    if direct.exists():
        return direct
    hits = sorted(root.rglob(filename))
    if not hits:
        raise FileNotFoundError(
            f"'{filename}' not found anywhere under {root} - "
            f"run notebook {notebook_hint} first; it is the notebook that writes this file."
        )
    return hits[0]


NODES_CSV = stage_file("02", "nodes.csv", "02_graph_construction")
EDGES_CSV = stage_file("02", "edges.csv", "02_graph_construction")
METRICS_CSV = stage_file("04", "stop_metrics.csv", "04_centrality_analysis")

nodes_df = pd.read_csv(NODES_CSV, dtype={"stop_id": str}, encoding="utf-8-sig")
edges_df = pd.read_csv(EDGES_CSV, dtype={"from_stop": str, "to_stop": str}, encoding="utf-8-sig")

print(f"nodes.csv       : {len(nodes_df):,} rows  <- {NODES_CSV}")
print(f"edges.csv       : {len(edges_df):,} rows  <- {EDGES_CSV}")
print(f"stop_metrics.csv: found            <- {METRICS_CSV}")

ATTR = {
    str(r["stop_id"]): {
        "stop_name": ("" if pd.isna(r.get("stop_name")) else str(r.get("stop_name"))),
        "lat": (np.nan if pd.isna(r.get("lat")) else float(r.get("lat"))),
        "lon": (np.nan if pd.isna(r.get("lon")) else float(r.get("lon"))),
        "region": ("" if pd.isna(r.get("region")) else str(r.get("region"))),
        "metro": ("" if pd.isna(r.get("metro")) else str(r.get("metro"))),
    }
    for r in nodes_df.to_dict("records")
}
DEFAULT_ATTR = {"stop_name": "", "lat": np.nan, "lon": np.nan, "region": "", "metro": ""}

# Directed segment frequencies from stage 02, keyed exactly as edges.csv stores them.
FREQ = {(str(r["from_stop"]), str(r["to_stop"])): int(r["trip_frequency"])
        for r in edges_df.to_dict("records")}
print(f"directed segments with a known trip_frequency: {len(FREQ):,}")

## 5. Parsing GTFS times - the >= 24:00 convention

This is the single most dangerous detail in the notebook, so it gets its own section and its own
tests.

GTFS clock fields are **not** wall-clock times. A trip that starts at 23:50 and runs past midnight is
written as `23:50:00`, `24:05:00`, `25:30:00`, ... - the hour field keeps counting past 23 so that the
whole trip stays inside one *service day*. In this feed roughly **1.1%** of rows have an hour field of
24 or more.

Two consequences:

1. **`datetime.strptime` cannot parse these values - it raises.** So we never use it.
2. If you were to normalise the hour modulo 24 (`25:30` -> `01:30`) to make `strptime` happy, the
   subtraction `arrival(v) - departure(u)` would silently return **-22.3 hours** instead of
   +7 minutes for every segment that crosses midnight. That is not a crash, it is a wrong number,
   which is worse.

The correct representation is **seconds since service midnight**: `int(h)*3600 + int(m)*60 + int(s)`,
with `h` allowed to exceed 23. Differences between two such values inside the same trip are then
always correct, including across midnight, because both endpoints live on the same monotone axis.

`gtfs_seconds` returns `None` (not a crash, not a zero) for blank or malformed values, so missing
times are counted as a discard reason rather than silently becoming 00:00:00. The assertions below
lock the behaviour in - including the exact midnight-crossing case that motivates the whole section.

In [ ]:
def gtfs_seconds(value):
    """GTFS 'HH:MM:SS' -> seconds since *service* midnight. Hours >= 24 are legal.

    Returns None for blank or malformed input. Never uses datetime/strptime, which
    raises on '25:30:00' - see the markdown above.
    """
    if not value:
        return None
    s = value.strip()
    if not s:
        return None
    parts = s.split(":")
    if len(parts) != 3:
        return None
    try:
        h, m, sec = int(parts[0]), int(parts[1]), int(parts[2])
    except ValueError:
        return None
    return h * 3600 + m * 60 + sec


# --- lock the contract with tests -----------------------------------------
assert gtfs_seconds("00:00:00") == 0
assert gtfs_seconds("05:12:23") == 5 * 3600 + 12 * 60 + 23
assert gtfs_seconds("25:30:00") == 25 * 3600 + 30 * 60          # 91800, NOT 5400
assert gtfs_seconds(" 5:07:00") == 5 * 3600 + 7 * 60            # single-digit hour
assert gtfs_seconds("") is None and gtfs_seconds(None) is None
assert gtfs_seconds("not a time") is None

# The midnight-crossing case that a naive parser gets catastrophically wrong.
depart_u = gtfs_seconds("23:58:00")
arrive_v = gtfs_seconds("24:06:00")
assert arrive_v - depart_u == 8 * 60, "midnight-crossing segment must be +8 minutes"
naive_wrong = (gtfs_seconds("00:06:00") - depart_u) / 60.0
print(f"service-midnight parsing : {(arrive_v - depart_u) / 60:.0f} min  (correct)")
print(f"naive hour-mod-24 parsing: {naive_wrong:.0f} min  (silently wrong)")
print("gtfs_seconds passes all contract tests.")

## 6. External data dependency: `stop_times.txt`

`stop_times.txt` is 816 MB / 15.7M rows - far above GitHub's file-size limit - so it is **not** in the
repository. The cell below downloads it from Google Drive on first run and skips the download if the
file is already present. This is the only external network dependency of the notebook. On a first
Colab run the download takes a few minutes.

In [ ]:
# stop_times.txt is 816MB and is not tracked in git - fetch it on demand.
_ensure("gdown")
import gdown
STOP_TIMES = DATA / "stop_times.txt"
if not STOP_TIMES.exists():
    gdown.download(id="1V_yPAWXV6mGTFGrfiosah5LngcLZnviW",
                   output=str(STOP_TIMES), quiet=False)
print("stop_times.txt:", round(STOP_TIMES.stat().st_size / 1024**2, 1), "MB")

## 7. The streaming pass: one duration distribution per segment

This is the expensive step - **3-6 minutes** - and the reason the rest of the project can stay cheap:
every later notebook reads `edges_traveltime.csv` instead of re-reading 816 MB.

The pass reuses the ordering assumption that notebook 02 states and *verifies*: the feed is sorted by
`(trip_id, stop_sequence)`, so two consecutive rows of the same `trip_id` are two consecutive stops of
that trip. We re-check it here for free by counting `stop_sequence` regressions and interleaved trip
blocks while we are already touching every row; if either counter is non-zero the durations below are
meaningless and the cell says so loudly.

For each consecutive pair `u -> v` inside a trip we compute

```
travel_seconds = arrival_time(v) - departure_time(u)
```

both parsed to seconds since service midnight. If `arrival_time` is blank we fall back to
`departure_time` on the same row and vice versa, which is harmless in this feed because the two
columns are identical on every row (verified by the `rows_with_dwell` counter below) - the Israeli
feed does not model dwell time at all. That is worth stating plainly: **our travel times are
door-to-door schedule differences and contain no separable dwell component**, so a segment's duration
is "time from leaving u to arriving at v" and nothing else.

Four discard rules, each counted separately so section 8 can report exactly what was thrown away:

| Rule | Why |
|---|---|
| `u == v` | a self-loop, a timing artefact in the feed; notebook 02 drops these from the graph too |
| either time missing/malformed | no duration can be computed |
| `duration < MIN_TRAVEL_SECONDS` (i.e. <= 0) | a non-positive travel time is impossible; it means the two rows are mis-ordered or share a timestamp |
| `duration > MAX_TRAVEL_SECONDS` (3 h) | no single scheduled leg between two consecutive stops in Israel takes three hours; these are feed errors, typically a trip whose times restart |

**Memory.** Durations are accumulated per segment in `array("i")` (4 bytes per observation) rather than
Python lists (28+ bytes per int). ~15.3M observations across ~52k segments is therefore roughly
**60 MB of payload, ~150 MB peak** including the numpy copies made in the next section. This is the
one place where the notebook is memory-hungry, and it is bounded by the number of rows in the feed.

In [ ]:
def stream_segment_durations(path, progress_every=PROGRESS_EVERY):
    """One pass over stop_times.txt collecting the travel-time distribution per segment.

    Returns (durations, stats) where durations maps (u, v) -> array('i') of observed
    scheduled travel times in seconds. Memory is O(observations), never O(rows) in
    Python objects.
    """
    durations = defaultdict(lambda: array("i"))
    stats = dict(
        rows_read=0, trips_seen=0, segments_seen=0, kept=0,
        drop_self_loop=0, drop_missing_time=0, drop_nonpositive=0, drop_too_long=0,
        rows_missing_time=0, rows_with_dwell=0, rows_hour_ge_24=0,
        stop_sequence_regressions=0, interleaved_trip_blocks=0,
    )
    seen_trips = set()
    t0 = time.time()

    with open(path, encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)
        header = next(reader)
        ti = header.index("trip_id")
        si = header.index("stop_id")
        ai = header.index("arrival_time")
        di = header.index("departure_time")
        qi = header.index("stop_sequence") if "stop_sequence" in header else None

        prev_trip = prev_stop = prev_dep = prev_seq = None
        for row in reader:
            stats["rows_read"] += 1
            trip = row[ti]
            stop = row[si]
            arr_raw, dep_raw = row[ai], row[di]
            arr = gtfs_seconds(arr_raw)
            dep = gtfs_seconds(dep_raw)
            if arr is None and dep is None:
                stats["rows_missing_time"] += 1
            elif arr is not None and dep is not None and arr != dep:
                stats["rows_with_dwell"] += 1
            if arr is not None and arr >= 24 * 3600:
                stats["rows_hour_ge_24"] += 1
            # Fall back across the two columns; in this feed they are identical.
            arr_eff = arr if arr is not None else dep
            dep_eff = dep if dep is not None else arr

            if trip != prev_trip:
                if trip in seen_trips:
                    stats["interleaved_trip_blocks"] += 1
                seen_trips.add(trip)
                prev_seq = None
            else:
                stats["segments_seen"] += 1
                if prev_stop == stop:
                    stats["drop_self_loop"] += 1
                elif prev_dep is None or arr_eff is None:
                    stats["drop_missing_time"] += 1
                else:
                    d = arr_eff - prev_dep
                    if d < MIN_TRAVEL_SECONDS:
                        stats["drop_nonpositive"] += 1
                    elif d > MAX_TRAVEL_SECONDS:
                        stats["drop_too_long"] += 1
                    else:
                        durations[(prev_stop, stop)].append(d)
                        stats["kept"] += 1

            if qi is not None:
                try:
                    seq = int(row[qi])
                except (ValueError, IndexError):
                    seq = None
                if seq is not None and prev_seq is not None and seq <= prev_seq:
                    stats["stop_sequence_regressions"] += 1
                prev_seq = seq

            prev_trip, prev_stop, prev_dep = trip, stop, dep_eff

            if progress_every and stats["rows_read"] % progress_every == 0:
                print(f"    {stats['rows_read']:,} rows | {len(durations):,} segments "
                      f"| {time.time() - t0:,.0f}s")

    stats["trips_seen"] = len(seen_trips)
    stats["distinct_segments"] = len(durations)
    stats["elapsed_seconds"] = round(time.time() - t0, 1)
    return durations, stats


print("Streaming stop_times.txt (~15.7M rows; 3-6 minutes) ...")
durations, stream_stats = stream_segment_durations(STOP_TIMES)

print("\nStreaming statistics:")
for k, v in stream_stats.items():
    print(f"  {k:<28} {v:,}" if isinstance(v, int) else f"  {k:<28} {v}")

if stream_stats["stop_sequence_regressions"] or stream_stats["interleaved_trip_blocks"]:
    print("\n" + "!" * 78)
    print("WARNING: the feed is NOT sorted by (trip_id, stop_sequence).")
    print("Consecutive rows are then not consecutive stops, so every duration above is")
    print("meaningless. Sort the file first (see notebook 02) before trusting this stage.")
    print("!" * 78)
else:
    print("\nOrdering assumption holds over the full file - durations are trustworthy.")

## 8. What was discarded, and why

Being explicit about discards is the difference between a filtered dataset and a quietly biased one.
The `assert` below is the point of this section: the table accounts for **every** consecutive stop pair
the streaming pass saw - each one either became an observation or fell into exactly one discard
bucket, and the two must add up exactly.

The counter to watch is `drop_nonpositive`. It fires when two consecutive stops of a trip carry the
**same timestamp**, which happens because the timetable is only specified to the minute on part of the
network: two stops served inside the same minute produce a difference of exactly 0 seconds. Those are
not corrupt rows, they are a *resolution limit* of the source data. On a 400k-row slice of this feed
the rate was about 0.5% of pairs; the printed table gives the true full-file figure, and that is the
number to quote rather than this one. The handling and its bias:

* We discard them rather than clamping them to 1 second, because a fabricated 1-second edge would
  make a chain of closely-spaced stops look like a free teleport to any shortest-path algorithm.
* A segment survives as long as **at least one** trip over it produced a positive duration, so
  discarding zero-duration observations shifts that segment's median *upward* slightly. Segments where
  *every* observation was zero vanish from the travel-time graph entirely - and those will be
  disproportionately the short, dense, urban links, i.e. the bias is not uniform across the network.
  Section 9 prints how many stage-02 segments were lost this way.

The bar chart uses a logarithmic axis because the kept bucket is orders of magnitude larger than the
others.

In [ ]:
buckets = {
    "kept (valid observation)": stream_stats["kept"],
    "self-loop (u == v)": stream_stats["drop_self_loop"],
    "missing / malformed time": stream_stats["drop_missing_time"],
    f"non-positive (< {MIN_TRAVEL_SECONDS}s)": stream_stats["drop_nonpositive"],
    f"absurd (> {MAX_TRAVEL_SECONDS // 3600}h)": stream_stats["drop_too_long"],
}
total_pairs = stream_stats["segments_seen"]
assert sum(buckets.values()) == total_pairs, "discard buckets must account for every pair"

discard_df = pd.DataFrame({
    "reason": list(buckets.keys()),
    "observations": list(buckets.values()),
})
discard_df["share_of_pairs"] = (discard_df["observations"] / total_pairs).round(6)
discard_df.to_csv(TABLES / "discard_reasons.csv", index=False, encoding="utf-8-sig")

print(f"consecutive stop pairs seen : {total_pairs:,}")
print(f"kept as observations        : {stream_stats['kept']:,} "
      f"({stream_stats['kept'] / total_pairs:.2%})")
print(f"rows where arrival != departure (dwell modelled): {stream_stats['rows_with_dwell']:,}")
print(f"rows with hour >= 24 (service-day convention)   : {stream_stats['rows_hour_ge_24']:,} "
      f"({stream_stats['rows_hour_ge_24'] / stream_stats['rows_read']:.2%})")
display(discard_df)

fig, ax = plt.subplots(figsize=(9, 4.5))
colors = ["#16a34a"] + ["#dc2626"] * (len(discard_df) - 1)
bars = ax.bar(range(len(discard_df)), discard_df["observations"], color=colors)
ax.set_yscale("log")
ax.set_xticks(range(len(discard_df)))
ax.set_xticklabels(discard_df["reason"], rotation=20, ha="right", fontsize=9)
ax.set_ylabel("Consecutive stop pairs (log scale)")
ax.set_title("Fate of every consecutive stop pair in stop_times.txt")
for b, v in zip(bars, discard_df["observations"]):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height(), f"{v:,}",
            ha="center", va="bottom", fontsize=8)
ax.margins(y=0.25)
fig.tight_layout()
fig.savefig(FIGURES / "discard_reasons.png", dpi=FIG_DPI)
plt.show()

## 9. From distributions to edge weights

Each segment now owns a full distribution of observed scheduled durations. We summarise it with three
order statistics:

* **median** - the edge weight. Median rather than mean because the distribution is right-skewed
  (a handful of trips are scheduled with long layovers) and one outlier should not move a weight.
* **p25 / p75** - the interquartile range, i.e. how *reliable* that edge is. A segment whose p25 and
  p75 are 4 and 6 minutes is a consistent link; one with 2 and 25 minutes means the schedule varies
  enormously by time of day or by route, and any single-number weight for it is a simplification.
  Exporting the quartiles means notebooks 20-22 can quantify that uncertainty instead of pretending
  the median is exact.

We also record `n_observations` (how many valid trips produced the estimate) next to `trip_frequency`
from notebook 02 (how many trips traverse the segment in total). The gap between the two columns *is*
the discard rate for that specific edge, so a reader can spot an edge whose median rests on three
surviving observations out of four hundred trips.

The **undirected** graph weight is computed by pooling the raw observations of both travel directions
and taking the median of the pooled set - not by averaging the two directional medians. Pooling is
exact and automatically weights each direction by how often it is actually served.

`edges_traveltime.csv` is written directed (one row per `from_stop -> to_stop`), matching the
orientation of `edges.csv` from notebook 02; the pickled graph is undirected, matching
`graph_undirected.pkl`. Both conventions are what the downstream notebooks expect.

In [ ]:
records = []
pooled = defaultdict(list)     # (min_id, max_id) -> list of numpy arrays

for (u, v), arr in durations.items():
    if len(arr) < MIN_OBSERVATIONS:
        continue
    a = np.asarray(arr, dtype=np.int32)
    p25, med, p75 = np.percentile(a, [25, 50, 75])
    records.append({
        "from_stop": u,
        "to_stop": v,
        "median_travel_seconds": float(round(med, 1)),
        "p25_travel_seconds": float(round(p25, 1)),
        "p75_travel_seconds": float(round(p75, 1)),
        "trip_frequency": int(FREQ.get((u, v), len(arr))),
        "n_observations": int(len(arr)),
    })
    pooled[(u, v) if u <= v else (v, u)].append(a)

tt_edges = pd.DataFrame.from_records(records)
tt_edges = tt_edges[["from_stop", "to_stop", "median_travel_seconds",
                     "p25_travel_seconds", "p75_travel_seconds",
                     "trip_frequency", "n_observations"]]
tt_edges.to_csv(TABLES / "edges_traveltime.csv", index=False, encoding="utf-8-sig")

# How many stage-02 segments could not be given a travel time at all?
missing_directed = sorted(set(FREQ) - set(zip(tt_edges["from_stop"], tt_edges["to_stop"])))
print(f"directed segments with a travel time : {len(tt_edges):,}")
print(f"stage-02 segments with NO valid observation: {len(missing_directed):,} "
      f"({len(missing_directed) / max(len(FREQ), 1):.2%} of stage-02 edges)")
print(f"total observations behind the table  : {tt_edges['n_observations'].sum():,}")
print(f"median observations per segment      : {tt_edges['n_observations'].median():.0f}")
print(f"segments resting on a single observation: "
      f"{(tt_edges['n_observations'] == 1).sum():,}")
display(tt_edges.head(TOP_N))

### Building and saving the travel-time graph

The undirected projection pools both directions, attaches the station attributes from notebook 02,
and stores three edge attributes:

* `travel_seconds` - the pooled median, the quantity that means something;
* `weight` - **a mirror of `travel_seconds`**, so that any `networkx` call that defaults to
  `weight="weight"` does time-based routing rather than frequency-based routing. This is a deliberate
  break from `graph_undirected.pkl`, where `weight` is the trip frequency. Downstream notebooks that
  load `graph_traveltime.pkl` are getting a *time* graph and must treat `weight` as seconds;
* `trip_frequency` - the stage-02 frequency, summed over both directions, kept so that a notebook
  needing both quantities does not have to re-join the CSV.

`del durations` frees the ~60 MB of accumulated arrays before the betweenness sections, which are the
next memory consumer.

In [ ]:
Gt = nx.Graph()
for (a, b), arrs in pooled.items():
    all_obs = np.concatenate(arrs)
    med = float(np.median(all_obs))
    freq = int(FREQ.get((a, b), 0)) + int(FREQ.get((b, a), 0))
    Gt.add_edge(a, b,
                travel_seconds=med,
                weight=med,                 # mirror: nx defaults to weight="weight"
                trip_frequency=freq,
                n_observations=int(all_obs.size))

for n in Gt.nodes():
    Gt.nodes[n].update(ATTR.get(n, DEFAULT_ATTR))

del durations, pooled

with open(STAGE / "graph_traveltime.pkl", "wb") as f:
    pickle.dump(Gt, f)

components = sorted(nx.connected_components(Gt), key=len, reverse=True)
Gc = Gt.subgraph(components[0]).copy()
lcc_share = Gc.number_of_nodes() / Gt.number_of_nodes()

print(f"travel-time graph : {Gt.number_of_nodes():,} nodes, {Gt.number_of_edges():,} edges")
print(f"connected components: {len(components):,}; "
      f"largest holds {Gc.number_of_nodes():,} nodes ({lcc_share:.2%})")
print(f"saved: {STAGE / 'graph_traveltime.pkl'} "
      f"({(STAGE / 'graph_traveltime.pkl').stat().st_size / 1024**2:.1f} MB)")

# Explicit comparison against the hop graph of notebook 02.
hop_nodes = set(nodes_df["stop_id"].astype(str))
print(f"\nstage-02 hop graph nodes: {len(hop_nodes):,}")
print(f"nodes lost to the travel-time filter: {len(hop_nodes - set(Gt.nodes())):,} "
      "(all of their segments had zero or unusable durations)")

## 10. What do the travel times look like?

Three sanity views before we trust the weights:

1. **Distribution of median travel time per segment** (log y-axis). A transit network should be
   dominated by short urban hops of one to three minutes with a thin tail of intercity legs. Anything
   else - a mode at zero, a bimodal spike at the 3-hour cap - would mean the parsing or the filters
   are wrong.
2. **Relative interquartile spread**, `(p75 - p25) / median`, which says how much a segment's
   scheduled duration varies across trips. Values near zero mean the timetable treats the segment as
   fixed; large values mean a single median is a poor summary of that edge.
3. **Travel time against service frequency**, on log-log axes: do the heavily-served segments tend to
   be the short urban ones? This is the check that the two weightings really are measuring different
   things - if travel time were just a monotone function of frequency, the whole re-weighting exercise
   would be redundant.

In [ ]:
med_s = tt_edges["median_travel_seconds"].to_numpy(dtype=float)
rel_iqr = ((tt_edges["p75_travel_seconds"] - tt_edges["p25_travel_seconds"])
           / tt_edges["median_travel_seconds"].replace(0, np.nan)).to_numpy(dtype=float)

pct = np.percentile(med_s, [1, 25, 50, 75, 90, 99])
print("Median travel time per segment (seconds): "
      f"p1={pct[0]:.0f}  p25={pct[1]:.0f}  p50={pct[2]:.0f}  "
      f"p75={pct[3]:.0f}  p90={pct[4]:.0f}  p99={pct[5]:.0f}")
print(f"in minutes: p50={pct[2] / 60:.1f}  p90={pct[4] / 60:.1f}  p99={pct[5] / 60:.1f}")
print(f"segments over 30 minutes: {(med_s > 1800).sum():,} "
      f"({(med_s > 1800).mean():.2%})")
print(f"median relative IQR (p75-p25)/median: {np.nanmedian(rel_iqr):.2f}")
print(f"segments with a zero-width IQR (identical on every trip): "
      f"{(np.nan_to_num(rel_iqr) == 0).sum():,}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

axes[0].hist(med_s / 60.0, bins=80, range=(0, 60), color="#2563eb",
             edgecolor="white", linewidth=0.3)
axes[0].set_yscale("log")
axes[0].set_xlabel("Median travel time (minutes, truncated at 60)")
axes[0].set_ylabel("Number of segments (log scale)")
axes[0].set_title("Segment travel-time distribution")

axes[1].hist(rel_iqr[np.isfinite(rel_iqr)], bins=60, range=(0, 3), color="#d97706",
             edgecolor="white", linewidth=0.3)
axes[1].set_yscale("log")
axes[1].set_xlabel("(p75 - p25) / median")
axes[1].set_ylabel("Number of segments (log scale)")
axes[1].set_title("How stable is each segment's schedule?")

sample = tt_edges.sample(min(20000, len(tt_edges)), random_state=PATH_SEED)
axes[2].scatter(sample["trip_frequency"].clip(lower=1),
                sample["median_travel_seconds"].clip(lower=1),
                s=3, alpha=0.15, color="#7c3aed")
axes[2].set_xscale("log")
axes[2].set_yscale("log")
axes[2].set_xlabel("Trip frequency (trips/day, both endpoints as ordered)")
axes[2].set_ylabel("Median travel time (s)")
axes[2].set_title("Frequency vs travel time")

fig.tight_layout()
fig.savefig(FIGURES / "traveltime_distribution.png", dpi=FIG_DPI)
plt.show()

rho_freq_time = (tt_edges["trip_frequency"].corr(tt_edges["median_travel_seconds"],
                                                 method="spearman"))
print(f"\nSpearman(trip_frequency, median_travel_seconds) = {rho_freq_time:.3f} "
      "- if this were near +/-1 the re-weighting would add nothing.")

## 11. Do travel-time shortest paths differ from hop-count shortest paths?

This is the first of the two tests that decide whether the earlier notebooks need a caveat.

We draw `N_PATH_PAIRS` random origin-destination pairs from the largest component and, for each one,
compute two routes **on the same graph**:

* the **hop-count** shortest path (unweighted BFS) - what every notebook so far implicitly assumed a
  passenger takes;
* the **travel-time** shortest path (Dijkstra on `travel_seconds`) - what a passenger minimising
  journey time would actually take.

Then we score them against each other:

* `same_path` - are the two routes literally the same sequence of stations?
* `hop_path_seconds` vs `time_path_seconds` - **how much time a passenger loses by following the
  hop-optimal route.** This is the number that quantifies the cost of the earlier modelling choice.
* `time_path_hops` vs `hop_path_hops` - how many extra stops the time-optimal route is willing to
  pass through to save those minutes.

Cost: roughly 1-3 minutes for 300 pairs. Both queries are bidirectional searches, so each is far
cheaper than a full single-source run, but Dijkstra in pure-Python `networkx` is still the slow half.

In [ ]:
rng = random.Random(PATH_SEED)
lcc_nodes = list(Gc.nodes())

pairs = []
while len(pairs) < N_PATH_PAIRS:
    s, t = rng.choice(lcc_nodes), rng.choice(lcc_nodes)
    if s != t:
        pairs.append((s, t))


def path_seconds(graph, path):
    return sum(graph[a][b]["travel_seconds"] for a, b in zip(path, path[1:]))


t0 = time.time()
rows = []
for s, t in pairs:
    hop_path = nx.shortest_path(Gc, s, t)                              # BFS
    time_path = nx.shortest_path(Gc, s, t, weight="travel_seconds")    # Dijkstra
    rows.append({
        "source": s, "target": t,
        "hop_path_hops": len(hop_path) - 1,
        "time_path_hops": len(time_path) - 1,
        "hop_path_seconds": round(path_seconds(Gc, hop_path), 1),
        "time_path_seconds": round(path_seconds(Gc, time_path), 1),
        "same_path": hop_path == time_path,
    })
print(f"{len(rows)} origin-destination pairs routed twice in {time.time() - t0:.1f}s")

paths = pd.DataFrame(rows)
paths["extra_seconds_if_hop_routed"] = paths["hop_path_seconds"] - paths["time_path_seconds"]
paths["extra_hops_if_time_routed"] = paths["time_path_hops"] - paths["hop_path_hops"]
paths["time_penalty_ratio"] = (paths["hop_path_seconds"]
                               / paths["time_path_seconds"].replace(0, np.nan))
paths.to_csv(TABLES / "path_comparison.csv", index=False, encoding="utf-8-sig")

same_share = paths["same_path"].mean()
med_penalty = paths["extra_seconds_if_hop_routed"].median()
mean_penalty = paths["extra_seconds_if_hop_routed"].mean()
med_ratio = paths["time_penalty_ratio"].median()
p90_penalty = paths["extra_seconds_if_hop_routed"].quantile(0.90)

print(f"\nidentical routes                        : {same_share:.1%} of pairs")
print(f"median time lost by hop-routing         : {med_penalty / 60:.1f} min")
print(f"mean time lost by hop-routing           : {mean_penalty / 60:.1f} min")
print(f"90th percentile time lost               : {p90_penalty / 60:.1f} min")
print(f"median hop-path / time-path duration    : {med_ratio:.2f}x")
print(f"median extra stops on the time-optimal route: "
      f"{paths['extra_hops_if_time_routed'].median():.0f}")
print(f"pairs where the time-optimal route is LONGER in hops: "
      f"{(paths['extra_hops_if_time_routed'] > 0).mean():.1%}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
lim = max(paths["hop_path_seconds"].max(), paths["time_path_seconds"].max()) / 60 * 1.05
axes[0].scatter(paths["time_path_seconds"] / 60, paths["hop_path_seconds"] / 60,
                s=14, alpha=0.5, color="#2563eb")
axes[0].plot([0, lim], [0, lim], ls="--", lw=1, color="#334155",
             label="identical duration")
axes[0].set_xlim(0, lim); axes[0].set_ylim(0, lim)
axes[0].set_xlabel("Time-optimal route (minutes)")
axes[0].set_ylabel("Hop-optimal route (minutes)")
axes[0].set_title("Cost of routing by hop count\n(points above the line = time wasted)")
axes[0].legend()

axes[1].hist(paths["extra_hops_if_time_routed"], bins=range(
    int(paths["extra_hops_if_time_routed"].min()) - 1,
    int(paths["extra_hops_if_time_routed"].max()) + 2),
    color="#16a34a", edgecolor="white", linewidth=0.4)
axes[1].set_xlabel("Extra stops on the time-optimal route")
axes[1].set_ylabel("Number of origin-destination pairs")
axes[1].set_title("How many more stops a time-minimising passenger accepts")

fig.tight_layout()
fig.savefig(FIGURES / "path_comparison.png", dpi=FIG_DPI)
plt.show()

## 12. Betweenness on the travel-time graph - and an honest control

Betweenness is the metric the project uses to call a station "critical", so it is the metric that
matters here. We recompute it with `travel_seconds` as the edge weight, which makes `networkx` run a
weighted (Dijkstra) shortest-path count instead of a BFS.

**The control matters more than the headline.** Comparing our new estimate directly against notebook
04's `approx_betweenness` would confound two different things: the change of weighting, and the fact
that the two runs sampled *different* source nodes (notebook 04 used `k=300`; sampled betweenness is
noisy, and notebook 04's own seed-stability check quantified that noise). So we run **three** numbers:

| Column | What it is |
|---|---|
| `bt_traveltime` | weighted betweenness, `k=K_BETWEENNESS`, seed `BETWEENNESS_SEED` |
| `bt_hop_control` | **unweighted** betweenness on the *same graph*, the *same k*, the *same seed* - therefore the same sampled source set. Any difference from `bt_traveltime` is caused purely by the weighting |
| `bt_hop_nb04` | notebook 04's published `approx_betweenness` (k=300, different sample, and computed on the stage-02 hop graph, which has slightly more edges) |

The clean scientific comparison is `bt_traveltime` vs `bt_hop_control`. `bt_hop_nb04` is reported
because it is what the earlier notebooks and the written report actually used, so its agreement -
or lack of it - is the practically relevant number.

**Cost: 5-15 minutes for the weighted run plus 1-3 minutes for the control.** Set `RUN_BETWEENNESS`
to `False` to skip sections 12-14; the contract outputs of this notebook
(`edges_traveltime.csv`, `graph_traveltime.pkl`) are already written at this point.

In [ ]:
bt_tt = bt_hop = None
k_eff = None

if RUN_BETWEENNESS:
    k_eff = min(K_BETWEENNESS, Gc.number_of_nodes())
    print(f"LCC: {Gc.number_of_nodes():,} nodes / {Gc.number_of_edges():,} edges | "
          f"k = {k_eff} sources ({k_eff / Gc.number_of_nodes():.2%} of the LCC)")

    t0 = time.time()
    bt_tt = nx.betweenness_centrality(Gc, k=k_eff, seed=BETWEENNESS_SEED,
                                      normalized=True, weight="travel_seconds")
    print(f"  travel-time (weighted) betweenness : {time.time() - t0:,.0f}s")

    t0 = time.time()
    bt_hop = nx.betweenness_centrality(Gc, k=k_eff, seed=BETWEENNESS_SEED,
                                       normalized=True, weight=None)
    print(f"  hop-count control (same k, same seed, same sources): {time.time() - t0:,.0f}s")

    nz_tt = sum(1 for v in bt_tt.values() if v > 0)
    nz_hop = sum(1 for v in bt_hop.values() if v > 0)
    print(f"\nnon-zero estimates: travel-time {nz_tt:,} | hop control {nz_hop:,} "
          f"of {len(bt_tt):,} stations")
    print("Stations at zero were simply never on a sampled shortest path - with k = "
          f"{k_eff} sources that is expected, not a finding.")
else:
    print("RUN_BETWEENNESS is False - sections 12-14 are skipped. "
          "The contract outputs of this notebook are already written.")

## 13. Does the ranking survive the re-weighting?

Now the actual question. We join the three betweenness columns on `stop_id` and report, for each pair:

* **Spearman rho over all LCC stations** - the rank correlation including the large mass of stations
  that scored zero under both estimators. Ties at zero inflate this number, so it is the optimistic
  reading.
* **Spearman rho over stations positive under either estimator** - the honest reading, because it
  measures agreement where there is actually something to disagree about.
* **top-50 and top-10 overlap** - the practically relevant number. The project's claims are about a
  shortlist of critical stations, so what matters is whether that shortlist is the same list.

Interpretation guide, stated in advance so it cannot be fitted after the fact: a top-50 overlap above
roughly 0.8 means the earlier hop-based shortlist is broadly safe; between 0.5 and 0.8 means it is
directionally right but individual station claims are fragile; below 0.5 means the hop-count ranking
and the travel-time ranking are substantially different objects and every earlier "most critical
station" statement needs the caveat attached.

In [ ]:
if RUN_BETWEENNESS:
    nb04 = pd.read_csv(METRICS_CSV, dtype={"stop_id": str}, encoding="utf-8-sig")
    if "approx_betweenness" not in nb04.columns:
        raise KeyError("stop_metrics.csv has no 'approx_betweenness' column - "
                       "re-run notebook 04_centrality_analysis.")
    nb04_bt = dict(zip(nb04["stop_id"], nb04["approx_betweenness"].astype(float)))

    bt = pd.DataFrame({
        "stop_id": list(Gc.nodes()),
    })
    bt["stop_name"] = bt["stop_id"].map(lambda n: Gc.nodes[n].get("stop_name", ""))
    bt["region"] = bt["stop_id"].map(lambda n: Gc.nodes[n].get("region", ""))
    bt["lat"] = bt["stop_id"].map(lambda n: Gc.nodes[n].get("lat", np.nan))
    bt["lon"] = bt["stop_id"].map(lambda n: Gc.nodes[n].get("lon", np.nan))
    bt["bt_traveltime"] = bt["stop_id"].map(bt_tt).fillna(0.0)
    bt["bt_hop_control"] = bt["stop_id"].map(bt_hop).fillna(0.0)
    bt["bt_hop_nb04"] = bt["stop_id"].map(nb04_bt).fillna(0.0)
    bt["rank_traveltime"] = bt["bt_traveltime"].rank(ascending=False, method="min").astype(int)
    bt["rank_hop_control"] = bt["bt_hop_control"].rank(ascending=False, method="min").astype(int)
    bt["rank_hop_nb04"] = bt["bt_hop_nb04"].rank(ascending=False, method="min").astype(int)
    bt["rank_shift_vs_control"] = bt["rank_hop_control"] - bt["rank_traveltime"]
    bt["rank_shift_vs_nb04"] = bt["rank_hop_nb04"] - bt["rank_traveltime"]
    bt = bt.sort_values("bt_traveltime", ascending=False).reset_index(drop=True)
    bt.to_csv(TABLES / "betweenness_traveltime.csv", index=False, encoding="utf-8-sig")

    missing_in_nb04 = int((~bt["stop_id"].isin(set(nb04_bt))).sum())

    def agreement(col_a, col_b, label):
        rho_all = bt[col_a].corr(bt[col_b], method="spearman")
        sub = bt[(bt[col_a] > 0) | (bt[col_b] > 0)]
        rho_pos = sub[col_a].corr(sub[col_b], method="spearman") if len(sub) > 2 else np.nan
        out = {"comparison": label, "n_stations": len(bt),
               "n_positive_either": len(sub),
               "spearman_all": round(float(rho_all), 4),
               "spearman_positive_either": round(float(rho_pos), 4)}
        for n in (10, 50, 100):
            a = set(bt.nlargest(n, col_a)["stop_id"])
            b = set(bt.nlargest(n, col_b)["stop_id"])
            out[f"top{n}_overlap"] = round(len(a & b) / n, 3)
        return out

    agree = pd.DataFrame([
        agreement("bt_traveltime", "bt_hop_control",
                  "travel-time vs hop control (same sample - clean)"),
        agreement("bt_traveltime", "bt_hop_nb04",
                  "travel-time vs notebook 04 (different sample - practical)"),
        agreement("bt_hop_control", "bt_hop_nb04",
                  "hop control vs notebook 04 (sampling noise floor)"),
    ])
    agree.to_csv(TABLES / "betweenness_agreement.csv", index=False, encoding="utf-8-sig")
    print(f"stations in the travel-time LCC missing from notebook 04: {missing_in_nb04:,}")
    display(agree.T)
else:
    bt = agree = None

### Reading the third row before the first two

The bottom row of that table - **hop control vs notebook 04** - is the *noise floor*. Both are
hop-count betweenness; they differ only because they sampled different sources (and because notebook
04 ran on the slightly larger stage-02 graph). Whatever disagreement that row shows is the amount of
disagreement sampling alone produces.

The re-weighting only tells us something if the travel-time rows disagree **more** than the noise
floor does. If all three rows look alike, the correct conclusion is "we cannot separate the effect of
weighting from estimator noise at k = 200", and the fix would be a larger `K_BETWEENNESS`, not a
stronger claim. The scatter plots below show the same three comparisons on ranks, where the noise is
easier to see than on the raw values.

In [ ]:
if RUN_BETWEENNESS:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5.2))
    comparisons = [
        ("rank_hop_control", "rank_traveltime", "Hop control rank", "Travel-time rank",
         "Same sample, only the weighting differs", "#2563eb"),
        ("rank_hop_nb04", "rank_traveltime", "Notebook 04 rank", "Travel-time rank",
         "What the earlier notebooks used", "#dc2626"),
        ("rank_hop_nb04", "rank_hop_control", "Notebook 04 rank", "Hop control rank",
         "Sampling noise floor (both hop-count)", "#94a3b8"),
    ]
    pool = bt.nsmallest(RANK_POOL, "rank_traveltime")
    for ax, (xc, yc, xl, yl, title, color) in zip(axes, comparisons):
        ax.scatter(bt[xc], bt[yc], s=4, alpha=0.15, color=color)
        ax.scatter(pool[xc], pool[yc], s=12, alpha=0.7, color="#111827",
                   label=f"top {RANK_POOL} by travel time")
        lim = len(bt)
        ax.plot([1, lim], [1, lim], ls="--", lw=1, color="#334155")
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_xlabel(xl); ax.set_ylabel(yl)
        ax.set_title(title, fontsize=11)
        ax.legend(fontsize=8, loc="lower right")
    fig.suptitle("Betweenness rank agreement (log-log; closer to the diagonal = more agreement)")
    fig.tight_layout()
    fig.savefig(FIGURES / "betweenness_rank_agreement.png", dpi=FIG_DPI)
    plt.show()

## 14. The biggest rank movers

Aggregate correlations hide the interesting cases, so we name them. The mover analysis is restricted
to the pool of stations ranked in the top `RANK_POOL` under **either** weighting: outside that pool
almost every station has a betweenness of exactly zero under a `k = 200` sample, and its "rank" is an
arbitrary tie-break rather than a measurement. Ranking movers over the whole 30k stations would
produce a table of pure noise.

`rank_shift_vs_control` is `rank_hop_control - rank_traveltime`, so:

* **positive** = the station is *more* central once minutes replace hops. These should be interchanges
  on fast corridors - rail stations and intercity terminals, where one edge covers a lot of ground
  quickly, so time-minimising paths funnel through them.
* **negative** = the station was an artefact of hop counting. These should be ordinary stops on long
  chains of closely-spaced urban stops, which look like efficient shortcuts when every edge costs 1
  but are slow in reality.

If the movers do **not** fall into those two families, the re-weighting is producing noise rather than
signal and the table should be read as such.

In [ ]:
if RUN_BETWEENNESS:
    in_pool = (bt["rank_traveltime"] <= RANK_POOL) | (bt["rank_hop_control"] <= RANK_POOL)
    movers = bt.loc[in_pool].copy()
    movers["abs_shift"] = movers["rank_shift_vs_control"].abs()
    movers = movers.sort_values("abs_shift", ascending=False)

    cols = ["stop_id", "stop_name", "region", "bt_traveltime", "bt_hop_control",
            "rank_traveltime", "rank_hop_control", "rank_shift_vs_control",
            "rank_shift_vs_nb04"]
    movers[cols + ["abs_shift"]].to_csv(
        TABLES / "betweenness_rank_movers.csv", index=False, encoding="utf-8-sig")

    print(f"mover pool: {len(movers):,} stations (top {RANK_POOL} under either weighting)")
    print(f"median |rank shift| inside the pool: "
          f"{movers['abs_shift'].median():.0f} places")
    print("\nRose most under travel-time weighting (gained importance):")
    display(movers.nlargest(TOP_N, "rank_shift_vs_control")[cols])
    print("Fell most under travel-time weighting (were hop-count artefacts):")
    display(movers.nsmallest(TOP_N, "rank_shift_vs_control")[cols])

    top_movers = movers.nlargest(2 * TOP_N, "abs_shift").sort_values("rank_shift_vs_control")
    labels = [f"{(n or sid)} ({sid})" for n, sid
              in zip(top_movers["stop_name"], top_movers["stop_id"])]
    colors = ["#dc2626" if v < 0 else "#16a34a"
              for v in top_movers["rank_shift_vs_control"]]

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.barh(range(len(top_movers)), top_movers["rank_shift_vs_control"], color=colors)
    ax.set_yticks(range(len(top_movers)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.axvline(0, color="#111827", lw=1)
    ax.set_xlabel("Rank shift (hop-count rank - travel-time rank);  "
                  "positive = more central under travel time")
    ax.set_title(f"Biggest betweenness rank movers (top {RANK_POOL} pool, same source sample)")
    fig.tight_layout()
    fig.savefig(FIGURES / "betweenness_rank_movers.png", dpi=FIG_DPI)
    plt.show()

### Where the two shortlists sit geographically

A last visual check: the top-50 station shortlist under each weighting, plotted on the country. If the
travel-time shortlist pulls towards rail corridors and intercity terminals while the hop-count
shortlist sits inside dense urban stop chains, that is the geographic signature of the effect
described above, and it makes the numeric result interpretable rather than just a correlation.

In [ ]:
if RUN_BETWEENNESS:
    geo = bt.dropna(subset=["lat", "lon"])
    top_tt = geo.nsmallest(50, "rank_traveltime")
    top_hop = geo.nsmallest(50, "rank_hop_control")
    shared = set(top_tt["stop_id"]) & set(top_hop["stop_id"])

    fig, ax = plt.subplots(figsize=(8, 11))
    ax.scatter(geo["lon"], geo["lat"], s=2, alpha=0.12, color="#94a3b8",
               label="all stations")
    ax.scatter(top_hop["lon"], top_hop["lat"], s=55, facecolors="none",
               edgecolors="#dc2626", linewidths=1.4, label="top 50 - hop count")
    ax.scatter(top_tt["lon"], top_tt["lat"], s=18, color="#2563eb",
               label="top 50 - travel time", zorder=5)
    ax.set_aspect(1 / np.cos(np.deg2rad(float(geo["lat"].mean()))))
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
    ax.set_title("Top-50 critical stations under each weighting\n"
                 f"({len(shared)} of 50 appear in both lists)")
    ax.legend(loc="lower right", fontsize=9)
    fig.tight_layout()
    fig.savefig(FIGURES / "top50_shortlist_map.png", dpi=FIG_DPI)
    plt.show()
    print(f"stations on both shortlists: {len(shared)} / 50")

## 15. Saving the stage summary

`traveltime_summary.json` is the machine-readable record of this stage: graph size, every discard
counter, the travel-time distribution percentiles, the path-comparison result and the full betweenness
agreement table. Notebooks 20-22 and the written report read it rather than re-deriving the numbers,
and it is the audit trail for the filtering decisions made in section 7.

In [ ]:
summary = {
    "stage": "18_travel_time_network",
    "edge_weight": "median scheduled travel time in seconds (attribute 'travel_seconds', "
                   "mirrored to 'weight')",
    "thresholds": {
        "min_travel_seconds": MIN_TRAVEL_SECONDS,
        "max_travel_seconds": MAX_TRAVEL_SECONDS,
        "min_observations": MIN_OBSERVATIONS,
    },
    "streaming": stream_stats,
    "discards": {k: int(v) for k, v in buckets.items()},
    "graph": {
        "nodes": Gt.number_of_nodes(),
        "undirected_edges": Gt.number_of_edges(),
        "directed_edges_with_traveltime": int(len(tt_edges)),
        "stage02_segments_without_traveltime": int(len(missing_directed)),
        "connected_components": len(components),
        "largest_component_nodes": Gc.number_of_nodes(),
        "largest_component_share": round(lcc_share, 4),
    },
    "travel_time_seconds": {
        "p1": float(pct[0]), "p25": float(pct[1]), "p50": float(pct[2]),
        "p75": float(pct[3]), "p90": float(pct[4]), "p99": float(pct[5]),
        "median_relative_iqr": float(round(np.nanmedian(rel_iqr), 4)),
        "spearman_frequency_vs_time": float(round(rho_freq_time, 4)),
    },
    "path_comparison": {
        "pairs": int(len(paths)),
        "identical_route_share": float(round(same_share, 4)),
        "median_seconds_lost_by_hop_routing": float(med_penalty),
        "mean_seconds_lost_by_hop_routing": float(round(mean_penalty, 1)),
        "p90_seconds_lost_by_hop_routing": float(p90_penalty),
        "median_duration_ratio": float(round(med_ratio, 4)),
        "median_extra_hops_on_time_route":
            float(paths["extra_hops_if_time_routed"].median()),
    },
    "betweenness": (
        {
            "k_samples": int(k_eff),
            "seed": BETWEENNESS_SEED,
            "agreement": agree.to_dict("records"),
            "median_abs_rank_shift_in_pool": float(movers["abs_shift"].median()),
            "rank_pool": RANK_POOL,
        }
        if RUN_BETWEENNESS else {"skipped": True}
    ),
}

with open(STAGE / "traveltime_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

written = [
    TABLES / "edges_traveltime.csv",
    STAGE / "graph_traveltime.pkl",
    STAGE / "traveltime_summary.json",
    TABLES / "discard_reasons.csv",
    TABLES / "path_comparison.csv",
]
if RUN_BETWEENNESS:
    written += [TABLES / "betweenness_traveltime.csv",
                TABLES / "betweenness_agreement.csv",
                TABLES / "betweenness_rank_movers.csv"]

print("Written:")
for p in written:
    print(f"  {p}  ({p.stat().st_size / 1024:,.0f} KB)")
print("\nSummary (graph + path comparison):")
print(json.dumps({k: summary[k] for k in ("graph", "path_comparison")},
                 indent=2, ensure_ascii=False))

## Takeaways

* **The project now has a time-weighted graph, and notebooks 20-22 depend on it.**
  `edges_traveltime.csv` gives every segment a median, p25 and p75 scheduled duration together with
  the number of observations behind it, and `graph_traveltime.pkl` carries the median as
  `travel_seconds` (mirrored to `weight`). Any downstream statement of the form "removing this station
  adds N minutes to the average journey" is only computable on this graph; on the hop graph it was not
  even definable.

* **Read the discard counters before trusting a single edge.** The discards here are not corruption,
  they are *resolution*: parts of the Israeli timetable are specified only to the minute, so some
  consecutive stop pairs carry identical timestamps and yield a duration of exactly zero. We drop
  those rather than clamp them, which biases every surviving median slightly **upward** and removes
  entirely those segments where no trip ever produced a positive duration - disproportionately the
  short, dense urban links, so the bias is not spread evenly over the network. Section 8 prints the
  exact counts and section 9 prints how many stage-02 segments were lost; those two numbers are the
  main limitation of the travel-time graph and should be quoted alongside any result derived from it.

* **Arrival and departure are identical on every row of this feed, so there is no dwell time.** Our
  "travel time" is purely the scheduled gap between leaving one stop and arriving at the next.
  Waiting at a stop, transferring between lines, and waiting for the next service are all invisible.
  A real passenger's journey time is therefore strictly longer than any number in this notebook, and
  the gap is largest exactly where transfers matter most - which is a bias against multimodal hubs.

* **Scheduled, not realised.** GTFS is a timetable. Congestion, bunching and delay are not in the
  data, so these are the times the operator *intends*, not the times a passenger *experiences*.

* **The path comparison quantifies the cost of the old assumption directly.** The printed
  "median time lost by hop-routing" is the number to cite: it is how much slower a hop-optimal route
  is than a time-optimal route on the same network, in minutes, over a random sample of
  origin-destination pairs. The share of pairs whose two routes are literally identical is the
  complementary statistic - where it is high, the earlier notebooks were accidentally right.

* **On the betweenness ranking, read the three-row agreement table, not one number.** The comparison
  that isolates the weighting effect is `travel-time vs hop control`, because both runs used the same
  `k` and the same seed and therefore the same sampled sources. The row `hop control vs notebook 04`
  is the noise floor: two hop-count estimates that differ only by sampling. **A change in the
  travel-time ranking is only evidence about weighting if it exceeds that floor.** If the top-50
  overlap between travel time and hop count is materially below the noise floor's overlap, then every
  earlier "most critical station" claim in this project is weighting-dependent and must carry that
  caveat; if it is not, the honest conclusion is that at `k = 200` we cannot separate the two effects,
  and the remedy is a larger `K_BETWEENNESS`, not a stronger claim.

* **Betweenness here is sampled and therefore noisy.** With `k = 200` sources out of ~30k stations,
  most stations score exactly zero simply because no sampled shortest path crossed them. That is why
  the rank-mover analysis is confined to the top-`RANK_POOL` pool, and why rank shifts for low-ranked
  stations are not reported at all - they would be an artefact of tie-breaking.

* **Weights are still supply, not demand.** As in every earlier stage, the graph describes the service
  the operator runs, not the passengers who use it. Travel time makes the supply model considerably
  more realistic; it does not make it a demand model. Notebook 21 is where a demand proxy enters.